In [3]:
import os
import polars as pl
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta
import time
from sqlalchemy import create_engine
import pymssql
from typing import List, Dict

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

import warnings
warnings.filterwarnings('ignore')

In [4]:
T,N = 3400, 5422
start_dt = '2008-01-01'     
end_dt = '2025-12-31'

zyyx_url = URL.create(drivername="mssql+pymssql",
             username="zyyxReader",
             password="zyyx!5893@Fund",
             host="10.110.0.106",
             database="zyyx",
             query={"charset": "utf8"})
# 如果需要在连接时指定 tds_version，可以这样处理
zyyx_engine = create_engine(
    zyyx_url,
    connect_args={
        "tds_version": "7.0",
        "charset": "utf8"
    }
)

zyyx_conn = zyyx_engine.connect()

In [7]:
dates = np.load('/data/xujiayi/end2end/axis/dates.npy', allow_pickle=True)
ticks = np.load('/data/xujiayi/end2end/axis/ticks.npy', allow_pickle=True)

close = np.memmap('/data/xujiayi/end2end/d_field/close.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
nan_mask = np.isnan(close)

In [6]:
sql = f"""
select 
    con_date as date, 
    stock_code as tick, 
    con_np_roll as con_np_ttm
from con_forecast_roll_stk
where con_date <='{end_dt}'
"""

con_np_ttm = pl.read_database(sql, zyyx_conn).sort(['tick','date'])
con_np_ttm = con_np_ttm.pivot(index='date', columns='tick', values='con_np_ttm').to_pandas().set_index('date')
con_np_ttm.index= pd.to_datetime(con_np_ttm.index)

total_dates = sorted(list(set(pd.to_datetime(dates)).union(set(con_np_ttm.index))))
con_np_ttm = con_np_ttm.reindex(index=total_dates)
con_np_ttm = con_np_ttm.ffill()
con_np_ttm = con_np_ttm.reindex(index=dates,columns=ticks)

con_np_ttm = con_np_ttm.values.astype(float)
con_np_ttm[nan_mask] = np.nan
con_np_ttm

array([[ 3.70707041e+05,  8.34084413e+05,  1.05375430e+01, ...,
                    nan,             nan,             nan],
       [ 3.56811191e+05,  8.38487928e+05,  1.05375430e+01, ...,
                    nan,             nan,             nan],
       [ 3.47910593e+05,  8.59725596e+05,  1.05375430e+01, ...,
                    nan,             nan,             nan],
       ...,
       [ 4.44003358e+06, -1.93140959e+06, -1.22799536e+04, ...,
                    nan,  2.49888767e+05,  6.27472703e+05],
       [ 4.44027808e+06, -1.92515479e+06, -1.22799536e+04, ...,
         3.51793322e+04,  2.50044384e+05,  6.27778482e+05],
       [ 4.44052258e+06, -1.91890000e+06, -1.22799536e+04, ...,
         3.51793322e+04,  2.50200000e+05,  6.25250028e+05]],
      shape=(4376, 5436))